Things go wrong at runtime — a file is missing, a user types letters where a number is expected, a network request times out. Python uses exceptions to signal these events. An exception interrupts normal execution and travels up the call stack until something catches it or the program crashes with a traceback. Understanding exceptions means writing programs that fail gracefully rather than abruptly, give useful error messages rather than cryptic tracebacks, and clean up resources reliably even when things go wrong. This notebook covers catching exceptions, raising them, chaining them, building custom exception types, and the design philosophy behind when to use each approach.

## The Exception Hierarchy

Every built-in exception is a class that inherits from `BaseException`. The ones you will deal with day-to-day inherit from `Exception`. A few critical ones (`KeyboardInterrupt`, `SystemExit`, `GeneratorExit`) inherit directly from `BaseException` — you almost never want to catch those by accident.

```text
BaseException
├── SystemExit
├── KeyboardInterrupt
├── GeneratorExit
└── Exception
    ├── ArithmeticError
    │   └── ZeroDivisionError
    ├── LookupError
    │   ├── IndexError
    │   └── KeyError
    ├── TypeError
    ├── ValueError
    ├── AttributeError
    ├── NameError
    ├── OSError  (also called IOError)
    │   ├── FileNotFoundError
    │   └── PermissionError
    └── RuntimeError
        └── RecursionError
```

Because exceptions are classes, catching a parent catches all its children too. Catching `LookupError` catches both `IndexError` and `KeyError`.

## Common Built-in Exceptions

Before writing any error handling, it helps to recognise the exceptions Python raises most often:

In [ ]:
# TypeError — wrong type for an operation
try:
    result = "3" + 5
except TypeError as e:
    print(f"TypeError: {e}")

# ValueError — right type, wrong value
try:
    n = int("hello")
except ValueError as e:
    print(f"ValueError: {e}")

# IndexError — index out of range
try:
    items = [1, 2, 3]
    print(items[10])
except IndexError as e:
    print(f"IndexError: {e}")

# KeyError — dict key not found
try:
    d = {"a": 1}
    print(d["z"])
except KeyError as e:
    print(f"KeyError: {e}")

# ZeroDivisionError
try:
    print(10 / 0)
except ZeroDivisionError as e:
    print(f"ZeroDivisionError: {e}")

# AttributeError — object has no such attribute
try:
    "hello".fly()
except AttributeError as e:
    print(f"AttributeError: {e}")

## `try` / `except`

The `try` block contains code that might raise an exception. If an exception occurs, Python skips the rest of the `try` block and looks for a matching `except` clause.

```python
try:
    # code that might fail
except SomeException:
    # what to do when it fails
```

Always catch the **most specific** exception you can. Catching `Exception` broadly is a last resort — it hides bugs and makes programs harder to debug.

In [ ]:
def parse_age(text):
    try:
        age = int(text)
        return age
    except ValueError:
        print(f"Could not parse {text!r} as an integer")
        return None

print(parse_age("28"))     # 28
print(parse_age("abc"))    # error message, then None
print(parse_age("3.5"))    # error message — int() won't parse floats

In [ ]:
# Use 'as' to bind the exception object — gives you the message and other details
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError as e:
        print(f"Cannot divide by zero: {e}")
        return None

print(safe_divide(10, 2))   # 5.0
print(safe_divide(10, 0))   # error message, then None

## Multiple `except` Clauses

A single `try` block can have multiple `except` clauses — Python tries each one in order and runs the first that matches. Put more specific exceptions before more general ones, or the general clause will shadow the specific ones.

In [ ]:
def lookup(data, key, index):
    try:
        return data[key][index]
    except KeyError:
        print(f"Key {key!r} not found")
    except IndexError:
        print(f"Index {index} out of range")
    except TypeError:
        print("data must be a dict of lists")
    return None

records = {"scores": [88, 74, 91]}

print(lookup(records, "scores", 1))   # 74
print(lookup(records, "grades", 0))   # KeyError
print(lookup(records, "scores", 9))   # IndexError
print(lookup(None, "scores", 0))      # TypeError

In [ ]:
# Catch multiple exceptions with a tuple — same handler for several types
def parse_number(text):
    try:
        return float(text)
    except (ValueError, TypeError) as e:
        print(f"Invalid input ({type(e).__name__}): {e}")
        return None

print(parse_number("3.14"))   # 3.14
print(parse_number("abc"))    # ValueError
print(parse_number(None))     # TypeError

## `else` and `finally`

The full `try` statement has four clauses:

```python
try:
    ...             # might raise
except SomeError:
    ...             # runs if SomeError was raised
else:
    ...             # runs only if NO exception was raised
finally:
    ...             # runs ALWAYS — exception or not
```

- **`else`** separates the happy path from error handling, making it clear what code only runs on success.
- **`finally`** is guaranteed to run even if an exception is raised and not caught, or if a `return` is executed inside the `try`. Use it to release resources — close files, release locks, disconnect from a database.

In [ ]:
def load_config(path):
    import json
    f = None
    try:
        f = open(path)
        data = json.load(f)
    except FileNotFoundError:
        print(f"Config file not found: {path}")
        return {}
    except json.JSONDecodeError as e:
        print(f"Invalid JSON in {path}: {e}")
        return {}
    else:
        # Only reaches here if open() and json.load() both succeeded
        print(f"Config loaded: {len(data)} keys")
        return data
    finally:
        # Always runs — close the file if it was opened
        if f:
            f.close()
            print("File closed")

result = load_config("nonexistent.json")

In [ ]:
# finally runs even when an exception propagates out
def risky():
    try:
        raise ValueError("something went wrong")
    finally:
        print("finally always runs — cleaning up")   # prints before the error propagates

try:
    risky()
except ValueError as e:
    print(f"Caught outside: {e}")

## Raising Exceptions

Use `raise` to signal an error from your own code. Pass an exception instance or an exception class (Python will instantiate it).

Raise an exception when:
- A precondition is violated — the caller passed bad input.
- An operation cannot be completed in a meaningful way.
- You want to convert a low-level exception to a higher-level one that makes more sense to the caller.

In [ ]:
def set_volume(level):
    if not isinstance(level, (int, float)):
        raise TypeError(f"Volume must be a number, got {type(level).__name__}")
    if not 0 <= level <= 100:
        raise ValueError(f"Volume must be between 0 and 100, got {level}")
    print(f"Volume set to {level}")

set_volume(75)          # OK

try:
    set_volume(150)
except ValueError as e:
    print(f"Error: {e}")

try:
    set_volume("loud")
except TypeError as e:
    print(f"Error: {e}")

In [ ]:
# Re-raise the current exception — preserve the original traceback
def process(data):
    try:
        result = 100 / data["divisor"]
    except KeyError:
        print("'divisor' key missing — logging and re-raising")
        raise   # bare raise re-raises the current exception unchanged
    return result

try:
    process({"value": 42})
except KeyError as e:
    print(f"Caller caught: {e}")

## Exception Chaining

When you catch one exception and raise another, Python records the relationship automatically (the original is stored in `__context__`). To make the relationship explicit — or to suppress it — use `raise ... from`.

- `raise NewError(...) from original_error` — explicit chain; traceback shows both.
- `raise NewError(...) from None` — suppresses the original; only the new exception appears in the traceback.

In [ ]:
class ConfigError(Exception):
    pass

def load_port(config):
    try:
        return int(config["port"])
    except KeyError as e:
        # Wrap the low-level KeyError in a higher-level ConfigError
        raise ConfigError("'port' is required in config") from e
    except ValueError as e:
        raise ConfigError(f"'port' must be an integer: {config['port']!r}") from e

try:
    load_port({"host": "localhost"})
except ConfigError as e:
    print(f"ConfigError: {e}")
    print(f"Caused by:   {e.__cause__}")

try:
    load_port({"port": "abc"})
except ConfigError as e:
    print(f"ConfigError: {e}")

In [ ]:
# raise ... from None — clean public API hides implementation details
class NotFoundError(Exception):
    pass

DATABASE = {"user:1": "Alice", "user:2": "Bob"}

def get_user(user_id):
    try:
        return DATABASE[f"user:{user_id}"]
    except KeyError:
        # Don't expose the internal key format to callers
        raise NotFoundError(f"User {user_id} not found") from None

try:
    get_user(99)
except NotFoundError as e:
    print(e)   # User 99 not found — no KeyError traceback visible

## Custom Exceptions

Define your own exception classes by subclassing `Exception` (or a more specific built-in). Custom exceptions let callers catch errors from your library specifically, without accidentally catching unrelated exceptions.

Conventions:
- Name ends in `Error` (e.g. `ValidationError`, `AuthError`).
- Create a base exception for your library/module, then subclass it for specific cases — callers can catch all library errors with one `except` clause.
- Keep them lightweight; add attributes only when the caller genuinely needs them to respond differently.

In [ ]:
# Base exception for a hypothetical payments module
class PaymentError(Exception):
    """Base class for all payment-related errors."""

class InsufficientFundsError(PaymentError):
    """Raised when the account balance is too low."""
    def __init__(self, balance, required):
        self.balance  = balance
        self.required = required
        super().__init__(
            f"Insufficient funds: need {required:.2f}, have {balance:.2f}"
        )

class CardDeclinedError(PaymentError):
    """Raised when the card is declined by the processor."""
    def __init__(self, reason="unknown"):
        self.reason = reason
        super().__init__(f"Card declined: {reason}")


def charge(amount, balance):
    if amount > balance:
        raise InsufficientFundsError(balance=balance, required=amount)
    return balance - amount

# Caller can catch the specific error and use its attributes
try:
    new_balance = charge(150.00, 42.50)
except InsufficientFundsError as e:
    print(e)
    print(f"  Shortfall: {e.required - e.balance:.2f}")

# Or catch all payment errors with one clause
try:
    raise CardDeclinedError("expired card")
except PaymentError as e:
    print(f"Payment failed: {e}")

## EAFP vs LBYL

There are two philosophies for dealing with potential errors:

**LBYL — Look Before You Leap**: check for the condition before attempting the operation.
```python
if key in d:
    value = d[key]
```

**EAFP — Easier to Ask Forgiveness than Permission**: just try the operation and handle the exception if it fails.
```python
try:
    value = d[key]
except KeyError:
    ...
```

Python's culture leans toward EAFP. The LBYL check and the actual operation are two separate steps — in concurrent code, the state can change between them (a race condition). EAFP is also faster when the happy path is common, because the check is free in the exception-free case. LBYL is fine for simple, non-concurrent situations where an `if` reads more naturally.

In [ ]:
import os

path = "data.csv"

# LBYL — check first, then act
if os.path.exists(path):
    with open(path) as f:
        content = f.read()
# Problem: file could be deleted between exists() check and open() call

# EAFP — just try it
try:
    with open(path) as f:
        content = f.read()
except FileNotFoundError:
    content = ""

# EAFP: type conversion without pre-checking
def to_int(value, default=0):
    try:
        return int(value)
    except (ValueError, TypeError):
        return default

print(to_int("42"))     # 42
print(to_int("oops"))   # 0
print(to_int(None))     # 0

## Exception Best Practices

A few rules that keep error handling from becoming a liability:

In [ ]:
# 1. Catch specific exceptions — never bare 'except:' or 'except Exception:' without good reason

# BAD — hides every possible error including bugs you'd want to know about
# try:
#     do_something()
# except:
#     pass

# GOOD — catch only what you know how to handle
try:
    value = int("42")
except ValueError:
    value = 0

print(value)

In [ ]:
# 2. Keep try blocks small — only wrap the lines that can actually raise

# BAD — too much in try, hard to know what raised
# try:
#     data = load_file(path)
#     processed = transform(data)
#     result = save(processed)
#     notify(result)
# except Exception:
#     ...

# GOOD — one concern per try block
def pipeline(path):
    try:
        data = open(path).read()
    except FileNotFoundError:
        return None

    # processing continues outside the try block
    lines = [line.strip() for line in data.splitlines() if line.strip()]
    return lines

print(pipeline("missing.txt"))   # None — not a crash

In [ ]:
# 3. Include context in error messages — what was the bad value?

def validate_email(email):
    if not isinstance(email, str):
        raise TypeError(f"Email must be a string, got {type(email).__name__!r}")
    if "@" not in email:
        raise ValueError(f"Invalid email address: {email!r}")
    return email.lower().strip()

try:
    validate_email("not-an-email")
except ValueError as e:
    print(e)   # Invalid email address: 'not-an-email'

try:
    validate_email(42)
except TypeError as e:
    print(e)   # Email must be a string, got 'int'

In [ ]:
# 4. Use logging instead of print for production error handling
import logging

logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s")

def fetch_user(user_id):
    try:
        if user_id <= 0:
            raise ValueError(f"user_id must be positive, got {user_id}")
        return {"id": user_id, "name": "Alice"}
    except ValueError:
        logging.warning("Invalid user_id: %s", user_id)
        return None

print(fetch_user(1))
print(fetch_user(-5))

## Summary

- Python exceptions are classes in a hierarchy rooted at `BaseException`. Catch `Exception` or its subclasses; avoid catching `BaseException` directly.
- **`try`/`except`**: wrap risky code in `try`; handle specific exceptions in `except`. Use `as e` to bind the exception object.
- **Multiple `except` clauses**: Python uses the first match — put specific exceptions before general ones. Catch multiple types in one clause with a tuple `except (A, B)`.
- **`else`** runs only when no exception was raised — separates the success path from error handling.
- **`finally`** runs always — exception or not, `return` or not. Use for cleanup: close files, release locks.
- **`raise`** signals an error from your own code. Bare `raise` re-raises the current exception. Choose the right built-in type (`ValueError` for bad values, `TypeError` for wrong types, etc.).
- **Exception chaining** (`raise New from original`) links a new exception to its cause. `raise New from None` suppresses the original traceback for clean public APIs.
- **Custom exceptions**: subclass `Exception`; create a library base class then specific subclasses. Name them with `Error`. Add attributes only when callers need them.
- **EAFP** (try it and handle failure) is preferred in Python over **LBYL** (check first) — it is race-condition-safe, faster on the happy path, and more idiomatic.
- Keep `try` blocks **small**. Catch **specific** exceptions. Include **context** in error messages. Use **logging** rather than `print` in production code.